In [ ]:
%pip install mediapipe

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
import math
import numpy as np

In [ ]:
#setting desired width and height of images
DESIRED_HEIGHT = 360
DESIRED_WIDTH = 360

In [ ]:
#function to resize and image show
def resize_and_show(image):
  h, w = image.shape[:2]
  if h < w:
    img = cv2.resize(image, (DESIRED_WIDTH, math.floor(h/(w/DESIRED_WIDTH))))
  else:
    img = cv2.resize(image, (math.floor(w/(h/DESIRED_HEIGHT)), DESIRED_HEIGHT))
  cv2_imshow(img)

In [ ]:
# Read images with OpenCV.
images = {name: cv2.imread(name) for name in uploaded.keys()}
# Preview the images.
for name, image in images.items():
  print(name)
  resize_and_show(image)

In [ ]:
import urllib.request

# Download the official selfie segmenter TFLite model from Google's servers
model_url = 'https://storage.googleapis.com/mediapipe-models/image_segmenter/selfie_segmenter/float16/latest/selfie_segmenter.tflite'
model_path = 'selfie_segmenter.tflite'

print("Downloading selfie segmenter model...")
urllib.request.urlretrieve(model_url, model_path)
print("Download complete! Model saved as 'selfie_segmenter.tflite'.")

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Configure the new Image Segmenter options
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.ImageSegmenterOptions(
    base_options=base_options,
    output_category_mask=True
)

print("MediaPipe Tasks Image Segmenter configured successfully!")

In [ ]:
# Show segmentation masks using modern ImageSegmenter
FIG_COLOR = (192, 0, 0) # Gray/Blue
MASK_COLOR = (0, 255, 255) # Yellow

# Open the modern segmenter using the options we defined
with vision.ImageSegmenter.create_from_options(options) as segmenter:
  for name, image in images.items():
    # Convert OpenCV BGR image to RGB, then wrap into a MediaPipe Image object
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    # Run segmentation
    segmentation_result = segmenter.segment(mp_image)
    # The category mask returned is shape (H, W, 1)
    category_mask = segmentation_result.category_mask.numpy_view()
    # Squeeze the (H, W, 1) mask to (H, W). [[1,2,3,4]] -> [1,2,3,4]
    category_mask_2d = np.squeeze(category_mask)

    # Generate solid color images for showing the output selfie segmentation mask
    fg_image = np.zeros(image.shape, dtype=np.uint8)
    fg_image[:] = MASK_COLOR
    bg_image = np.zeros(image.shape, dtype=np.uint8)
    bg_image[:] = FIG_COLOR

    # Stack along the 3rd axis to expand (H, W) to (H, W, 3)
    condition = np.stack((category_mask_2d, category_mask_2d, category_mask_2d), axis=-1) > 0.2
    output_image = np.where(condition, fg_image, bg_image)

    print(f'Segmentation mask of {name}:')
    resize_and_show(output_image)
    cv2.imwrite('segmentation_mask_' + str(name), output_image)

## Apply Background Blur

Use the `segmentation_result.category_mask` to identify the background of the `image` and apply a Gaussian blur effect to it. The mask will be inverted to blur only the background.

Itterate through the images, obtaining a segmentation mask, blurring the entire image, and then combining the unblurred foreground with the blurred background using the mask. You will need to convert the mask to a 3-channel array, and then multiply it with the original image. Then you can add the two images together.


In [ ]:
blurred_images = []

# Open the segmenter using the options
with vision.ImageSegmenter.create_from_options(options) as segmenter:
  for name, image in images.items():
    # Convert OpenCV BGR image to RGB, then wrap into a MediaPipe Image object
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    # Run segmentation
    segmentation_result = segmenter.segment(mp_image)
    category_mask = segmentation_result.category_mask.numpy_view()

    # Normalize the mask values to 0.0 - 1.0 range
    # Invert mask values here so 1.0 represents the foreground (person) and 0.0 represents the background
    # np.squeeze gets rid of unnecessary extra dimensions e.g. [[[1,2,3],[3,4,5]]] -> [[1,2,3],[3,4,5]]
    category_mask_normalized = 1.0 - (np.squeeze(category_mask).astype(float) / 255.0)

    # Apply Gaussian blur to the original image
    blurred_image = cv2.GaussianBlur(image, (55, 55), 0)

    # Convert segmentation mask to a 3-channel image
    segmentation_mask_3_channel = np.stack((category_mask_normalized, category_mask_normalized, category_mask_normalized), axis=-1)

    # Isolate the foreground (person stays sharp, because person is 1, background is 0)
    fg_image = (image * segmentation_mask_3_channel).astype(np.uint8)

    # Invert the segmentation mask to get the background mask
    inverted_mask_3_channel = 1.0 - segmentation_mask_3_channel

    # Isolate the blurred background
    bg_image = (blurred_image * inverted_mask_3_channel).astype(np.uint8)

    # Combine the sharp foreground and blurred background
    output_image = cv2.add(fg_image, bg_image)

    blurred_images.append(output_image)

    print(f'Image with blurred background for {name}:')
    resize_and_show(output_image)

## Make a composite image

Create a composite image by taking the person from the original image and placing them on a new background. Use the `segmentation_result.category_mask ` and the `image` from the previous section, extract only the foreground (person) to create a masked foreground layer. Add that to a new background.

In [ ]:
foreground_images = []
my_segmentation_mask_3_channel = None

# Open the segmenter using the options
with vision.ImageSegmenter.create_from_options(options) as segmenter:
  for name, image in images.items():
    # Convert OpenCV BGR image to RGB, then wrap into a MediaPipe Image object
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    # Run segmentation
    segmentation_result = segmenter.segment(mp_image)
    category_mask = segmentation_result.category_mask.numpy_view()

    # Normalize the mask values to 0.0 - 1.0 range
    # Invert mask values here so 1.0 represents the foreground (person) and 0.0 represents the background
    # np.squeeze gets rid of unnecessary extra dimensions e.g. [[[1,2,3],[3,4,5]]] -> [[1,2,3],[3,4,5]]
    category_mask_normalized = 1.0 - (np.squeeze(category_mask).astype(float) / 255.0)

    # Convert segmentation mask to a 3-channel image
    segmentation_mask_3_channel = np.stack((category_mask_normalized, category_mask_normalized, category_mask_normalized), axis=-1)
    my_segmentation_mask_3_channel = segmentation_mask_3_channel

    # Isolate the foreground (person) from the original image
    newfg_image = (image * segmentation_mask_3_channel).astype(np.uint8)

    foreground_images.append(newfg_image)

    print(f'Foreground image of {name}:')
    resize_and_show(newfg_image)


In [ ]:
from google.colab import files
bg_uploaded = files.upload() # i think this is irrelevant

In [ ]:
# Read images with OpenCV.
images_background = {name: cv2.imread(name) for name in bg_uploaded.keys()}
# Preview the images.
for name, image in images_background.items():
  print(name)
  resize_and_show(image)

In [ ]:
key = list(images_background.keys())[0]
resize_and_show(images_background[key])

In [ ]:
# Load an image and ensure it is resized to the same dimensions as the foreground image for proper overlay.
newbg_image = images_background[key]

# Get dimensions of the first foreground image (assuming all foregrounds have the same dimensions for this task)
foreground_height, foreground_width, _ = foreground_images[0].shape

# Resize the sith background to match the foreground dimensions
resized_newbg_image = cv2.resize(newbg_image, (foreground_width, foreground_height))

print('Resized Background:')
resize_and_show(resized_newbg_image)

In [ ]:
# take background and remove the foreground data in preparation for merging

# Invert the segmentation mask to get the background mask
inverted_mask_3_channel = 1.0 - my_segmentation_mask_3_channel

# Isolate the new background
isolatedresized_newbg_image = (resized_newbg_image * inverted_mask_3_channel).astype(np.uint8)
resize_and_show(isolatedresized_newbg_image)

# view our current foreground image
resize_and_show(newfg_image)

# Combine the masked foreground and new background
output_image = cv2.add(newfg_image, isolatedresized_newbg_image)

print(f'Image with blurred background for {name}:')
resize_and_show(output_image)

## Display a Facemesh

Use the mediapipe facemesh to showthe face landmarks on the image

In [ ]:
import urllib.request

# Download the MediaPipe Face Landmarker model file
face_landmarker_url = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task'
face_landmarker_path = 'face_landmarker.task'

print("Downloading face landmarker model...")
urllib.request.urlretrieve(face_landmarker_url, face_landmarker_path)
print("Download complete!")

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Configure Face Landmarker
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1
)

# Process the first uploaded image
image_key = list(images.keys())[0]
original_image = images[image_key].copy()
h, w, _ = original_image.shape
rgb_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

with vision.FaceLandmarker.create_from_options(options) as landmarker:
    detection_result = landmarker.detect(mp_image)

    # Draw face landmarks manually
    if detection_result.face_landmarks:
        print(f"Detected {len(detection_result.face_landmarks)} face(s). Plotting landmarks...")
        for face_landmarks in detection_result.face_landmarks:
            for landmark in face_landmarks:
                # Convert normalized coordinates to pixel coordinates
                cx, cy = int(landmark.x * w), int(landmark.y * h)
                # Draw a tiny circle on each landmark
                cv2.circle(original_image, (cx, cy), 2, (0, 255, 0), -1)


        print("Face landmarks successfully plotted on the image:")
        resize_and_show(original_image)
    else:
        print("No face detected in the image.")

In [ ]:
# Calculate bounding box of the detected face from the landmarks
if detection_result.face_landmarks:
    for face_landmarks in detection_result.face_landmarks:
        # Extract x and y coordinates in pixels
        xs = [int(landmark.x * w) for landmark in face_landmarks]
        ys = [int(landmark.y * h) for landmark in face_landmarks]

        # Find min/max values
        min_x, max_x = min(xs), max(xs)
        min_y, max_y = min(ys), max(ys)

        # Add a 50% padding to the crop box for better visualization
        box_w = max_x - min_x
        box_h = max_y - min_y

        pad_x = int(box_w * 0.5)
        pad_y = int(box_h * 0.5)

        # Clamp the coordinates to stay within image boundaries
        start_x = max(0, min_x - pad_x)
        end_x = min(w, max_x + pad_x)
        start_y = max(0, min_y - pad_y)
        end_y = min(h, max_y + pad_y)

        # Crop and show
        print("start_y:end_y, start_x:end_x", start_y, end_y, start_x, end_x)
        cropped_face = original_image[start_y:end_y, start_x:end_x]
        print("Cropped Face Details:")
        resize_and_show(cropped_face)
else:
    print("No face detected to crop.")